# Dataset Research: Foundations for ML, LP Constraints, and Simulation

This notebook explores the reduced Fannie Mae 2017 loan-level dataset to inform
the three modeling stages ahead. The goal is to understand the data well enough
to make grounded decisions in each stage:

- **Machine Learning:** target balance, feature distributions, and the risk
  gradient that a default model will need to capture.
- **Linear Programming constraints:** loan amount ranges to size the budget,
  and state and zip concentration to set diversification caps and the
  average-PD ceiling.
- **Simulation:** baseline default rates and risk structure that will feed the
  Monte Carlo comparison later.

All figures here use the historical default_flag as a stand-in for predicted
probability of default until the ML stage produces calibrated model output.

## Load Dataset and Check Size, Schema, and Target Balance

Loads the reduced loan-level file and confirms the basics before analysis:
row count, feature count, column dtypes, and the class balance of the target.
The default rate at the end gives a quick read on how imbalanced the problem is.

In [4]:
import polars as pl

df = pl.read_parquet("../data/processed/fannie_2017_loan_level.parquet")

# size, rows, features
print(f"Rows: {df.height:,}")
print(f"Features: {df.width}")
print(df.schema)

# target class imbalance 
target = "default_flag"
counts = df[target].value_counts().sort(target)
print(counts)
print(df[target].mean())  # default rate as a proportion

Rows: 2,046,851
Features: 115
Schema({'LOAN_ID': String, 'POOL_ID': String, 'ACT_PERIOD': String, 'CHANNEL': String, 'SELLER': String, 'SERVICER': String, 'MASTER_SERVICER': String, 'ORIG_RATE': String, 'CURR_RATE': String, 'ORIG_UPB': String, 'ISSUANCE_UPB': String, 'CURRENT_UPB': String, 'ORIG_TERM': String, 'ORIG_DATE': String, 'FIRST_PAY': String, 'LOAN_AGE': String, 'REM_MONTHS': String, 'ADJ_REM_MONTHS': String, 'MATR_DT': String, 'OLTV': String, 'OCLTV': String, 'NUM_BO': String, 'DTI': String, 'CSCORE_B': String, 'CSCORE_C': String, 'FIRST_FLAG': String, 'PURPOSE': String, 'PROP': String, 'NO_UNITS': String, 'OCC_STAT': String, 'STATE': String, 'MSA': String, 'ZIP': String, 'MI_PCT': String, 'PRODUCT': String, 'PPMT_FLG': String, 'IO': String, 'FIRST_PAY_IO': String, 'MNTHS_TO_AMTZ_IO': String, 'PMT_HISTORY': String, 'MOD_FLAG': String, 'MI_CANCEL_FLAG': String, 'ZB_DTE': String, 'LAST_UPB': String, 'RPRCH_DTE': String, 'CURR_SCHD_PRNCPL': String, 'TOT_SCHD_PRNCPL': String, 'UN

## Summary Statistics Across All Columns

A quick one-line overview of the full dataset. Gives count, null count, mean,
standard deviation, min, max, and quartiles for every column at once. Useful
for spotting ranges, missing data, and anything that looks off before deeper
analysis.

In [5]:
df.describe()

statistic,LOAN_ID,POOL_ID,ACT_PERIOD,CHANNEL,SELLER,SERVICER,MASTER_SERVICER,ORIG_RATE,CURR_RATE,ORIG_UPB,ISSUANCE_UPB,CURRENT_UPB,ORIG_TERM,ORIG_DATE,FIRST_PAY,LOAN_AGE,REM_MONTHS,ADJ_REM_MONTHS,MATR_DT,OLTV,OCLTV,NUM_BO,DTI,CSCORE_B,CSCORE_C,FIRST_FLAG,PURPOSE,PROP,NO_UNITS,OCC_STAT,STATE,MSA,ZIP,MI_PCT,PRODUCT,PPMT_FLG,…,RELOCATION_MORTGAGE_INDICATOR,ZERO_BALANCE_CODE_CHANGE_DATE,LOAN_HOLDBACK_INDICATOR,LOAN_HOLDBACK_EFFECTIVE_DATE,DELINQUENT_ACCRUED_INTEREST,PROPERTY_INSPECTION_WAIVER_INDICATOR,HIGH_BALANCE_LOAN_INDICATOR,ARM_5_YR_INDICATOR,ARM_PRODUCT_TYPE,MONTHS_UNTIL_FIRST_PAYMENT_RESET,MONTHS_BETWEEN_SUBSEQUENT_PAYMENT_RESET,INTEREST_RATE_CHANGE_DATE,PAYMENT_CHANGE_DATE,ARM_INDEX,ARM_CAP_STRUCTURE,INITIAL_INTEREST_RATE_CAP,PERIODIC_INTEREST_RATE_CAP,LIFETIME_INTEREST_RATE_CAP,MARGIN,BALLOON_INDICATOR,PLAN_NUMBER,FORBEARANCE_INDICATOR,HIGH_LOAN_TO_VALUE_HLTV_REFINANCE_OPTION_INDICATOR,DEAL_NAME,RE_PROCS_FLAG,ADR_TYPE,ADR_COUNT,ADR_UPB,PAYMENT_DEFERRAL_MOD_EVENT_FLAG,INTEREST_BEARING_UPB,ORIG_CLASSIC_FICO,ISSUE_CLASSIC_FICO,CURR_CLASSIC_FICO,max_dlq_ever,zero_bal_code,default_flag,orig_quarter
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str
"""count""","""2046851""","""0""","""2046851""","""2046851""","""2046851""","""2046851""","""0""","""2046851""","""2046851""","""2046851""","""0""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046511""","""2045278""","""963708""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""614740""","""2046851""","""2046851""",…,"""2046851""","""0""","""0""","""0""","""0""","""2046851""","""2046851""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""2046851""","""0""","""0""","""0""","""0""","""0""","""2046851""","""0""","""0""","""0""","""0""",2.046851e6,"""1528334""",2.046851e6,"""2046851"""
"""null_count""","""0""","""2046851""","""0""","""0""","""0""","""0""","""2046851""","""0""","""0""","""0""","""2046851""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""340""","""1573""","""1083143""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""1432111""","""0""","""0""",…,"""0""","""2046851""","""2046851""","""2046851""","""2046851""","""0""","""0""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""0""","""2046851""","""2046851""","""2046851""","""2046851""","""2046851""","""0""","""2046851""","""2046851""","""2046851""","""2046851""",0.0,"""518517""",0.0,"""0"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.635931,null,0.034143,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.760508,null,0.181597,null
"""min""","""100002130634""",null,"""012017""","""B""","""Amerihome Mortgage Company, Ll…","""1st 2nd Mortgage Company Of Ne…",null,"""1.790""","""1.790""","""10000.00""

## Quick Data Inspection Commands

A reference set of Polars commands for getting oriented in the dataset:
schema and dtypes, a transposed preview of values, null counts per column,
a row preview, and a frequency table for a single column. Useful as a first
pass before deeper analysis.

In [6]:
df.schema           # column names and dtypes, like df.dtypes
df.glimpse()        # transposed preview, close to pandas df.info() + a peek at values
df.null_count()     # nulls per column, one row back
df.head(10)         # same as pandas
df["default_flag"].value_counts()   # frequency table for a single column

Rows: 2046851
Columns: 115
$ LOAN_ID                                            <str> '123137870847', '102157127407', '118628664860', '102993029045', '144572067909', '138444152104', '112817980899', '126148139122', '109759244771', '149125240859'
$ POOL_ID                                            <str> null, null, null, null, null, null, null, null, null, null
$ ACT_PERIOD                                         <str> '012017', '022017', '032017', '022017', '022017', '032017', '012017', '012017', '032017', '012017'
$ CHANNEL                                            <str> 'C', 'R', 'R', 'R', 'R', 'B', 'B', 'C', 'R', 'R'
$ SELLER                                             <str> 'Caliber Home Loans, Inc.', 'Quicken Loans Inc.', 'Other', 'Other', 'Other', 'Loandepot.Com, Llc', 'Other', 'Wells Fargo Bank, N.A.', 'Quicken Loans Inc.', 'Truist Bank (Formerly Suntrust Bank)'
$ SERVICER                                           <str> 'Other', 'Quicken Loans Inc.', 'Other', 'Other', 'Other', 

default_flag,count
i8,u32
0,1976965
1,69886


## Loan Amount, State, and Zip Concentration Overview

This section establishes the baseline distribution facts needed to size the LP constraints. It covers three things: the loan amount summary (mean, median, and quartiles) to inform the budget figure, state-level concentration to inform the diversification cap, and zip-prefix concentration for finer geographic detail. Loan amounts are expressed as dollars and geographic shares as percentages of the full book, which matches the units the constraints will use.

In [7]:

# cast loan amount to float in case it came in as string
df = df.with_columns(pl.col("ORIG_UPB").cast(pl.Float64, strict=False))

# --- 1. Loan amount: average and range ---
loan_stats = df.select(
    pl.col("ORIG_UPB").mean().alias("mean_upb"),
    pl.col("ORIG_UPB").median().alias("median_upb"),
    pl.col("ORIG_UPB").min().alias("min_upb"),
    pl.col("ORIG_UPB").max().alias("max_upb"),
    pl.col("ORIG_UPB").quantile(0.25).alias("p25_upb"),
    pl.col("ORIG_UPB").quantile(0.75).alias("p75_upb"),
)
print("Loan amount summary:")
print(loan_stats)

# --- 2. State concentration ---
total = df.height
state_conc = (
    df.group_by("STATE")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / total * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
)
print(f"\nNumber of distinct states: {state_conc.height}")
print("State concentration (all, sorted):")
print(state_conc)

# --- 3. Zip concentration, top 10 ---
zip_conc = (
    df.group_by("ZIP")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / total * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
)
print(f"\nNumber of distinct zip prefixes: {zip_conc.height}")
print("Top 10 zip prefixes by concentration:")
print(zip_conc.head(10))

Loan amount summary:
shape: (1, 6)
┌───────────────┬────────────┬─────────┬─────────┬──────────┬──────────┐
│ mean_upb      ┆ median_upb ┆ min_upb ┆ max_upb ┆ p25_upb  ┆ p75_upb  │
│ ---           ┆ ---        ┆ ---     ┆ ---     ┆ ---      ┆ ---      │
│ f64           ┆ f64        ┆ f64     ┆ f64     ┆ f64      ┆ f64      │
╞═══════════════╪════════════╪═════════╪═════════╪══════════╪══════════╡
│ 228839.312192 ┆ 206000.0   ┆ 5000.0  ┆ 1.223e6 ┆ 137000.0 ┆ 300000.0 │
└───────────────┴────────────┴─────────┴─────────┴──────────┴──────────┘

Number of distinct states: 54
State concentration (all, sorted):
shape: (54, 3)
┌───────┬────────┬──────┐
│ STATE ┆ count  ┆ pct  │
│ ---   ┆ ---    ┆ ---  │
│ str   ┆ u32    ┆ f64  │
╞═══════╪════════╪══════╡
│ CA    ┆ 280415 ┆ 13.7 │
│ TX    ┆ 164662 ┆ 8.04 │
│ FL    ┆ 141082 ┆ 6.89 │
│ AZ    ┆ 74413  ┆ 3.64 │
│ IL    ┆ 74134  ┆ 3.62 │
│ …     ┆ …      ┆ …    │
│ AK    ┆ 3237   ┆ 0.16 │
│ VT    ┆ 2646   ┆ 0.13 │
│ PR    ┆ 2449   ┆ 0.12 │
│ VI    ┆

## State Concentration with Cumulative Share and 4% Cap Check

This section builds on the state concentration view by adding a running cumulative share, so you can see how much of the portfolio the top states hold together. It also flags how many states currently exceed a 4% cap, which tells you at a glance how many would be actively constrained if you set the diversification limit there.

In [8]:
state_conc = (
    df.group_by("STATE")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
    .with_columns(pl.col("pct").cum_sum().round(2).alias("cumulative_pct"))
)

print("Top 10 states with cumulative share:")
print(state_conc.head(10))

# how many states currently exceed a 4% cap
over_cap = state_conc.filter(pl.col("pct") > 4.0)
print(f"\nStates over 4%: {over_cap.height}")
print(over_cap)

Top 10 states with cumulative share:
shape: (10, 4)
┌───────┬────────┬──────┬────────────────┐
│ STATE ┆ count  ┆ pct  ┆ cumulative_pct │
│ ---   ┆ ---    ┆ ---  ┆ ---            │
│ str   ┆ u32    ┆ f64  ┆ f64            │
╞═══════╪════════╪══════╪════════════════╡
│ CA    ┆ 280415 ┆ 13.7 ┆ 13.7           │
│ TX    ┆ 164662 ┆ 8.04 ┆ 21.74          │
│ FL    ┆ 141082 ┆ 6.89 ┆ 28.63          │
│ AZ    ┆ 74413  ┆ 3.64 ┆ 32.27          │
│ IL    ┆ 74134  ┆ 3.62 ┆ 35.89          │
│ WA    ┆ 72077  ┆ 3.52 ┆ 39.41          │
│ CO    ┆ 71785  ┆ 3.51 ┆ 42.92          │
│ MI    ┆ 67644  ┆ 3.3  ┆ 46.22          │
│ NC    ┆ 65230  ┆ 3.19 ┆ 49.41          │
│ NY    ┆ 62924  ┆ 3.07 ┆ 52.48          │
└───────┴────────┴──────┴────────────────┘

States over 4%: 3
shape: (3, 4)
┌───────┬────────┬──────┬────────────────┐
│ STATE ┆ count  ┆ pct  ┆ cumulative_pct │
│ ---   ┆ ---    ┆ ---  ┆ ---            │
│ str   ┆ u32    ┆ f64  ┆ f64            │
╞═══════╪════════╪══════╪════════════════╡
│ CA    ┆ 28

## State Concentration and Risk

This section pairs geographic concentration with default rate at the state level. Sorting by portfolio share and showing each state's default rate side by side lets us see whether the largest states are also the riskiest, or whether concentration and risk move independently. That distinction matters for deciding how much work a state-level diversification cap does versus the average-PD ceiling.

In [9]:
state_risk = (
    df.group_by("STATE")
    .agg(
        pl.len().alias("count"),
        pl.col("default_flag").mean().alias("default_rate"),
    )
    .with_columns([
        (pl.col("count") / df.height * 100).round(2).alias("pct"),
        (pl.col("default_rate") * 100).round(2).alias("default_rate_pct"),
    ])
    .sort("pct", descending=True)
)

print("Top 10 states: concentration vs risk")
print(state_risk.select(["STATE", "pct", "default_rate_pct"]).head(10))

Top 10 states: concentration vs risk
shape: (10, 3)
┌───────┬──────┬──────────────────┐
│ STATE ┆ pct  ┆ default_rate_pct │
│ ---   ┆ ---  ┆ ---              │
│ str   ┆ f64  ┆ f64              │
╞═══════╪══════╪══════════════════╡
│ CA    ┆ 13.7 ┆ 3.38             │
│ TX    ┆ 8.04 ┆ 4.61             │
│ FL    ┆ 6.89 ┆ 5.69             │
│ AZ    ┆ 3.64 ┆ 2.99             │
│ IL    ┆ 3.62 ┆ 4.1              │
│ WA    ┆ 3.52 ┆ 2.43             │
│ CO    ┆ 3.51 ┆ 2.26             │
│ MI    ┆ 3.3  ┆ 2.6              │
│ NC    ┆ 3.19 ┆ 2.64             │
│ NY    ┆ 3.07 ┆ 5.62             │
└───────┴──────┴──────────────────┘


## Zip Prefix Concentration and Risk Against a 0.50% Cap

This section looks at geographic concentration at the zip-prefix level and pairs it with default rate, so we can see both dimensions together. Filtering to prefixes above a 0.50% cap shows how many would be constrained and whether those dense pockets are actually riskier, or just larger. That tells us whether a zip-level cap would do meaningful risk work or mainly serve diversification.

In [10]:
zip_risk = (
    df.group_by("ZIP")
    .agg(
        pl.len().alias("count"),
        pl.col("default_flag").mean().alias("default_rate"),
    )
    .with_columns([
        (pl.col("count") / df.height * 100).round(3).alias("pct"),
        (pl.col("default_rate") * 100).round(2).alias("default_rate_pct"),
    ])
    .sort("pct", descending=True)
)

# zip prefixes that exceed a 0.50% cap
over_cap = zip_risk.filter(pl.col("pct") > 0.50)
print(f"Zip prefixes over 0.50%: {over_cap.height} out of {zip_risk.height}")
print(over_cap.select(["ZIP", "pct", "default_rate_pct"]))

Zip prefixes over 0.50%: 32 out of 899
shape: (32, 3)
┌─────┬───────┬──────────────────┐
│ ZIP ┆ pct   ┆ default_rate_pct │
│ --- ┆ ---   ┆ ---              │
│ str ┆ f64   ┆ f64              │
╞═════╪═══════╪══════════════════╡
│ 750 ┆ 1.286 ┆ 4.34             │
│ 945 ┆ 1.101 ┆ 2.96             │
│ 852 ┆ 1.031 ┆ 2.27             │
│ 300 ┆ 0.982 ┆ 3.82             │
│ 840 ┆ 0.85  ┆ 1.96             │
│ …   ┆ …     ┆ …                │
│ 334 ┆ 0.522 ┆ 6.14             │
│ 554 ┆ 0.516 ┆ 2.91             │
│ 982 ┆ 0.511 ┆ 2.68             │
│ 275 ┆ 0.507 ┆ 2.19             │
│ 928 ┆ 0.504 ┆ 3.04             │
└─────┴───────┴──────────────────┘


## Baseline Default Rates: Book-Level and by FICO Band

Before setting an average-PD ceiling, it helps to establish two reference points from the actual data. First, the book-level default rate, both amount-weighted (the form the LP constraint uses) and unweighted, to confirm they agree. Second, the default rate across standard FICO bands, which shows the underlying risk gradient and gives a feel for how much safe supply is available to draw from.

Note: this uses default_flag (actual outcomes) as a stand-in until the ML model produces predicted PD.

In [11]:
# Overall weighted-average default rate (the whole book)
# Weighted by loan amount, since that is how the LP constraint works
overall = df.select(
    (
        (pl.col("default_flag") * pl.col("ORIG_UPB")).sum()
        / pl.col("ORIG_UPB").sum()
    ).alias("wtd_avg_default_rate")
)
print("Amount-weighted average default rate (full book):")
print(overall)

# Simple (unweighted) default rate for comparison
print(f"\nUnweighted default rate: {df['default_flag'].mean() * 100:.2f}%")

# What the rate looks like if you only funded the safer half, etc.
# Sort by risk proxy is not possible yet without PD, so show distribution
# across credit score bands as a rough risk gradient
#
# Nulls get their own band. Without this, a null CSCORE_B fails every
# comparison above and falls through to "Poor (below 580)", which silently
# fills that band with no-score loans instead of subprime borrowers.
band = (
    df.with_columns(pl.col("CSCORE_B").cast(pl.Int32, strict=False))
    .with_columns(
        pl.when(pl.col("CSCORE_B").is_null()).then(pl.lit("Unknown (no score)"))
        .when(pl.col("CSCORE_B") >= 800).then(pl.lit("Exceptional (800+)"))
        .when(pl.col("CSCORE_B") >= 740).then(pl.lit("Very Good (740-799)"))
        .when(pl.col("CSCORE_B") >= 670).then(pl.lit("Good (670-739)"))
        .when(pl.col("CSCORE_B") >= 580).then(pl.lit("Fair (580-669)"))
        .otherwise(pl.lit("Poor (below 580)"))
        .alias("fico_band")
    )
    .group_by("fico_band")
    .agg(
        pl.len().alias("count"),
        (pl.col("default_flag").mean() * 100).round(2).alias("default_rate_pct"),
    )
    .sort("default_rate_pct")
)
print("Default rate by standard FICO band:")
print(band)

Amount-weighted average default rate (full book):
shape: (1, 1)
┌──────────────────────┐
│ wtd_avg_default_rate │
│ ---                  │
│ f64                  │
╞══════════════════════╡
│ 0.034982             │
└──────────────────────┘

Unweighted default rate: 3.41%
Default rate by standard FICO band:
shape: (6, 3)
┌─────────────────────┬────────┬──────────────────┐
│ fico_band           ┆ count  ┆ default_rate_pct │
│ ---                 ┆ ---    ┆ ---              │
│ str                 ┆ u32    ┆ f64              │
╞═════════════════════╪════════╪══════════════════╡
│ Exceptional (800+)  ┆ 308739 ┆ 0.88             │
│ Very Good (740-799) ┆ 967786 ┆ 1.98             │
│ Unknown (no score)  ┆ 1573   ┆ 2.8              │
│ Good (670-739)      ┆ 623257 ┆ 5.31             │
│ Fair (580-669)      ┆ 145495 ┆ 10.25            │
│ Poor (below 580)    ┆ 1      ┆ 100.0            │
└─────────────────────┴────────┴──────────────────┘


## Achievable Average Prob of Default (PD) at Different Selection Levels

This section helps ground the average-PD ceiling in real numbers. By sorting
loans safest-first and tracking the running amount-weighted default rate as we
fund more of the book, we can see which ceiling values are actually reachable
before picking one. (Using default_flag as a PD stand-in until the ML model
produces predicted probabilities.)

In [12]:
# Using default_flag as the PD stand-in until the ML model exists.
# Sort loans safest-first, then see the running weighted-average default
# rate as you fund more and more of the book.

ranked = (
    df.select(["default_flag", "ORIG_UPB", "CSCORE_B"])
    .with_columns(pl.col("CSCORE_B").cast(pl.Int32, strict=False))
    .sort("CSCORE_B", descending=True)  # safest (highest FICO) first
    .with_columns([
        (pl.col("default_flag") * pl.col("ORIG_UPB")).alias("wtd_default"),
    ])
    .with_columns([
        pl.col("wtd_default").cum_sum().alias("cum_wtd_default"),
        pl.col("ORIG_UPB").cum_sum().alias("cum_upb"),
    ])
    .with_columns(
        (pl.col("cum_wtd_default") / pl.col("cum_upb") * 100).alias("running_avg_pd_pct")
    )
)

# Sample the running average at a few funding levels (share of book funded)
total = ranked.height
for frac in [0.10, 0.25, 0.50, 0.75, 1.00]:
    row = ranked.slice(0, int(total * frac)).tail(1)
    print(f"Funding safest {int(frac*100)}% of loans -> avg PD: "
          f"{row['running_avg_pd_pct'][0]:.2f}%")

Funding safest 10% of loans -> avg PD: 0.88%
Funding safest 25% of loans -> avg PD: 1.08%
Funding safest 50% of loans -> avg PD: 1.51%
Funding safest 75% of loans -> avg PD: 2.22%
Funding safest 100% of loans -> avg PD: 3.50%


## LGD Feasibility Check: Recovery Data on Defaulted Loans

Before computing LGD from the Qi-Yang formula, we need to know whether the data
can even support it. The formula requires recovery fields that only populate
after a loan defaults, forecloses, and sells. This checks how many defaulted
loans actually have those fields populated. That count decides whether the
Qi-Yang and hybrid approaches are viable, or whether the Sirignano flat value
is the honest fallback.

In [13]:
recovery_cols = [
    "NET_SALES_PROCEEDS",
    "FORECLOSURE_COSTS",
    "PROPERTY_PRESERVATION_AND_REPAIR_COSTS",
    "ASSET_RECOVERY_COSTS",
    "LAST_UPB",
]

defaulted = df.filter(pl.col("default_flag") == 1)
print(f"Total defaulted loans: {defaulted.height:,}")

# non-null count per recovery field among defaulted loans
for col in recovery_cols:
    non_null = defaulted.filter(pl.col(col).is_not_null()).height
    pct = non_null / defaulted.height * 100 if defaulted.height else 0
    print(f"{col:<45} {non_null:>7,} populated ({pct:.1f}%)")

Total defaulted loans: 69,886
NET_SALES_PROCEEDS                                  0 populated (0.0%)
FORECLOSURE_COSTS                                   0 populated (0.0%)
PROPERTY_PRESERVATION_AND_REPAIR_COSTS              0 populated (0.0%)
ASSET_RECOVERY_COSTS                                0 populated (0.0%)
LAST_UPB                                            0 populated (0.0%)


## Confirming Recovery Fields Are Empty Across the Full Book

A quick sanity check to rule out any chance the recovery fields were dropped or
renamed during reduction. If they are entirely null across all 2M loans, not
just the defaults, that confirms the fields simply do not carry into the
performance file, and the Qi-Yang formula genuinely cannot be applied here.

In [14]:
recovery_cols = [
    "NET_SALES_PROCEEDS",
    "FORECLOSURE_COSTS",
    "PROPERTY_PRESERVATION_AND_REPAIR_COSTS",
    "ASSET_RECOVERY_COSTS",
    "LAST_UPB",
]

print(df.select(recovery_cols).null_count())
print(f"\nTotal rows: {df.height:,}")

shape: (1, 5)
┌────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────┐
│ NET_SALES_PROCEEDS ┆ FORECLOSURE_COSTS ┆ PROPERTY_PRESERVATION ┆ ASSET_RECOVERY_COSTS ┆ LAST_UPB │
│ ---                ┆ ---               ┆ _AND_REPA…            ┆ ---                  ┆ ---      │
│ u32                ┆ u32               ┆ ---                   ┆ u32                  ┆ u32      │
│                    ┆                   ┆ u32                   ┆                      ┆          │
╞════════════════════╪═══════════════════╪═══════════════════════╪══════════════════════╪══════════╡
│ 2046851            ┆ 2046851           ┆ 2046851               ┆ 2046851              ┆ 2046851  │
└────────────────────┴───────────────────┴───────────────────────┴──────────────────────┴──────────┘

Total rows: 2,046,851


## Findings: LGD Cannot Be Computed From This Dataset

**Result:** All five recovery fields (NET_SALES_PROCEEDS, FORECLOSURE_COSTS,
PROPERTY_PRESERVATION_AND_REPAIR_COSTS, ASSET_RECOVERY_COSTS, LAST_UPB) are
null across all 2,046,851 loans, including all 69,886 defaulted loans.

**Implication:**
- The Qi-Yang (2007) formula is not computable here, since it requires realized
  recovery and expense figures that this performance file does not carry.
- The hybrid approach (deriving an average LGD from completed-sale loans) is
  also ruled out, as it depends on the same missing fields.

**Decision:** Adopt the Sirignano, Tsoukalas, and Giesecke (2016) flat-value
approach for LGD. Given the 2017 vintage sits in a stable housing period, a
normal-economy value (30 percent) is the baseline, with a downturn value
(50 percent) available for stress scenarios in the simulation stage.

**Justification:** This is an evidence-based fallback, not a shortcut. The
rigorous formula was tested against the data and found inapplicable due to
absent recovery fields, so the established peer-reviewed flat-value method is
the defensible alternative.

## LGD Scenarios: Normal (30%) vs Downturn (50%)

Applies the two Sirignano flat-value LGD assumptions to the book to see the
expected loss they imply per loan and in aggregate. This gives a concrete feel
for how much the choice of LGD matters before it feeds into expected return and
the simulation.

In [15]:
# LGD is the fraction of the loan balance lost when a loan defaults.
# Expected loss per loan = PD * LGD * loan amount.
# Using default_flag as the PD stand-in until the model produces predicted PD.

df = df.with_columns(pl.col("ORIG_UPB").cast(pl.Float64, strict=False))

for lgd in [0.30, 0.50]:
    result = df.select(
        (pl.col("default_flag") * lgd * pl.col("ORIG_UPB")).sum().alias("total_expected_loss"),
        (pl.col("default_flag") * lgd * pl.col("ORIG_UPB")).mean().alias("avg_loss_per_loan"),
    )
    total_loss = result["total_expected_loss"][0]
    avg_loss = result["avg_loss_per_loan"][0]
    total_upb = df["ORIG_UPB"].sum()
    loss_rate = total_loss / total_upb * 100
    print(f"LGD = {int(lgd*100)}%")
    print(f"  Total expected loss:      ${total_loss:,.0f}")
    print(f"  Avg loss per loan:        ${avg_loss:,.2f}")
    print(f"  Loss as % of total book:  {loss_rate:.2f}%")
    print()

LGD = 30%
  Total expected loss:      $4,915,608,900
  Avg loss per loan:        $2,401.55
  Loss as % of total book:  1.05%

LGD = 50%
  Total expected loss:      $8,192,681,500
  Avg loss per loan:        $4,002.58
  Loss as % of total book:  1.75%



## Findings: Expected Loss Under Normal and Downturn LGD

**Setup:** Applied the two Sirignano flat-value LGD assumptions to the full
book. Expected loss per loan = PD * LGD * loan amount, using default_flag as
the PD stand-in until the model produces predicted PD.

**Results:**

| Scenario | LGD | Total Expected Loss | Avg Loss / Loan | Loss as % of Book |
|----------|-----|---------------------|-----------------|-------------------|
| Normal   | 30% | $4.92B              | $2,401.55       | 1.05%             |
| Downturn | 50% | $8.19B              | $4,002.58       | 1.75%             |

**Takeaways:**
- Actual capital loss (1.05% to 1.75%) sits well below the 3.4% default rate,
  since only a fraction of each defaulted balance is lost after recovery.
- The LGD choice introduces about 0.70 points of spread in book-level loss,
  a modest but real sensitivity.

**Decision:** Use 30% as the baseline for primary analysis, matching the stable
2017 housing period. Reserve 50% as a downturn stress scenario for the
simulation stage.

## Interest Rate Distribution

Summarizes ORIG_RATE across the book. This drives the return side of the
expected-return calculation, so understanding its central tendency and spread
sets up the LP stage.

In [16]:
df = df.with_columns(pl.col("ORIG_RATE").cast(pl.Float64, strict=False))

rate_stats = df.select(
    pl.col("ORIG_RATE").mean().round(3).alias("mean_rate"),
    pl.col("ORIG_RATE").median().round(3).alias("median_rate"),
    pl.col("ORIG_RATE").min().alias("min_rate"),
    pl.col("ORIG_RATE").max().alias("max_rate"),
    pl.col("ORIG_RATE").quantile(0.25).round(3).alias("p25_rate"),
    pl.col("ORIG_RATE").quantile(0.75).round(3).alias("p75_rate"),
    pl.col("ORIG_RATE").std().round(3).alias("std_rate"),
)
print("Interest rate summary (%):")
print(rate_stats)

Interest rate summary (%):
shape: (1, 7)
┌───────────┬─────────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ mean_rate ┆ median_rate ┆ min_rate ┆ max_rate ┆ p25_rate ┆ p75_rate ┆ std_rate │
│ ---       ┆ ---         ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ f64       ┆ f64         ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞═══════════╪═════════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ 4.14      ┆ 4.125       ┆ 1.79     ┆ 6.125    ┆ 3.875    ┆ 4.5      ┆ 0.495    │
└───────────┴─────────────┴──────────┴──────────┴──────────┴──────────┴──────────┘


## Does Interest Rate Track Risk?

Groups loans by FICO band and shows the average interest rate alongside the
default rate for each. If rate rises as credit quality falls, it confirms the
risk-based pricing that the optimizer will navigate when trading return against
default risk.

In [26]:
rate_vs_risk = (
    df.with_columns(pl.col("CSCORE_B").cast(pl.Int32, strict=False))
    .with_columns(
        pl.when(pl.col("CSCORE_B").is_null()).then(pl.lit("Unknown (no score)"))
        .when(pl.col("CSCORE_B") >= 800).then(pl.lit("Exceptional (800+)"))
        .when(pl.col("CSCORE_B") >= 740).then(pl.lit("Very Good (740-799)"))
        .when(pl.col("CSCORE_B") >= 670).then(pl.lit("Good (670-739)"))
        .when(pl.col("CSCORE_B") >= 580).then(pl.lit("Fair (580-669)"))
        .otherwise(pl.lit("Poor (below 580)"))
        .alias("fico_band")
    )
    .group_by("fico_band")
    .agg(
        pl.len().alias("count"),
        pl.col("ORIG_RATE").mean().round(3).alias("avg_rate"),
        (pl.col("default_flag").mean() * 100).round(2).alias("default_rate_pct"),
    )
    .sort("avg_rate")
)
print("Interest rate vs default risk by FICO band:")
print(rate_vs_risk)

Interest rate vs default risk by FICO band:
shape: (6, 4)
┌─────────────────────┬────────┬──────────┬──────────────────┐
│ fico_band           ┆ count  ┆ avg_rate ┆ default_rate_pct │
│ ---                 ┆ ---    ┆ ---      ┆ ---              │
│ str                 ┆ u32    ┆ f64      ┆ f64              │
╞═════════════════════╪════════╪══════════╪══════════════════╡
│ Poor (below 580)    ┆ 1      ┆ 3.625    ┆ 100.0            │
│ Exceptional (800+)  ┆ 308739 ┆ 3.97     ┆ 0.88             │
│ Very Good (740-799) ┆ 967786 ┆ 4.064    ┆ 1.98             │
│ Unknown (no score)  ┆ 1573   ┆ 4.199    ┆ 2.8              │
│ Good (670-739)      ┆ 623257 ┆ 4.262    ┆ 5.31             │
│ Fair (580-669)      ┆ 145495 ┆ 4.491    ┆ 10.25            │
└─────────────────────┴────────┴──────────┴──────────────────┘


## Findings: Interest Rate Distribution and Risk-Based Pricing

**Rate distribution:**
- Rates are tightly clustered. Mean 4.14%, median 4.13%, with the middle 50%
  falling between 3.875% and 4.5% (std 0.495). Full range is 1.79% to 6.125%.
- The return side of the book varies little from loan to loan.

**Rate vs risk by FICO band** (Poor band holds a single loan and is excluded;
no-score loans are shown separately below):

| FICO Band            | Count   | Avg Rate | Default Rate |
|----------------------|---------|----------|--------------|
| Exceptional (800+)   | 308,739 | 3.97%    | 0.88%        |
| Very Good (740-799)  | 967,786 | 4.06%    | 1.98%        |
| Good (670-739)       | 623,257 | 4.26%    | 5.31%        |
| Fair (580-669)       | 145,495 | 4.49%    | 10.25%       |

**Key takeaway:**
- Pricing tracks risk, but weakly. From Exceptional to Fair, rate rises about
  0.5 points while default rate climbs more than elevenfold.
- The extra interest on riskier loans does not compensate for the added default
  risk. Safer loans likely carry higher expected return once loss is netted out.

**The no-score loans are worth keeping.** 1,573 loans have no FICO at all. They
priced at 4.199%, between Good and Fair, but defaulted at only 2.8%, better than
Good. Lenders treated them as moderately risky and they outperformed that. Small
group, but it argues for carrying them with a missing-score flag rather than
dropping them.

## Future Impact: Remaining Project

**ML stage:**
- FICO is a strong, clean predictor of default, which supports its weight as a
  feature. The steep risk gradient means the model has real signal to learn.
- Rate is priced off the same risk the model is trying to predict, and the near-flat
  rate gradient against an elevenfold default gradient shows how little independent
  signal it carries. This is the case for keeping ORIG_RATE out of the feature set
  and using it only for the LP income calculation.

**LP stage:**
- Expected return will likely favor high-FICO loans, since their low default
  cost barely dents a rate similar to riskier loans. The optimizer may lean
  toward the safest segment on its own.
- This is what makes the average-PD ceiling and diversification caps meaningful.
  Without them, the portfolio could pile into low-risk loans and lose realism.
  With them, the constraints force a more balanced, defensible allocation.

**Simulation stage:**
- The lopsided risk-return structure sets up a clear contrast between the
  optimized portfolio and the naive baseline.

### Primary sources worth citing:

* OCC Comptroller's Handbook, Concentrations of Credit (Version 2.0, October 2020)
* OCC Comptroller's Handbook, Loan Portfolio Management (April 1998)
* 12 CFR 32 and 12 USC 84 (national bank lending limits)
* FFIEC Interagency Guidance on Concentrations in Commercial Real Estate (2006)

## First-Time Homebuyer Share of the Book

Uses FIRST_FLAG (Y or N) to see how many loans went to first-time homebuyers,
and pairs it with default rate to check whether that group carries different
risk. This may inform whether first-time buyer status is worth including as a
feature or constraint dimension later.

In [18]:
fthb = (
    df.group_by("FIRST_FLAG")
    .agg(
        pl.len().alias("count"),
        (pl.col("default_flag").mean() * 100).round(2).alias("default_rate_pct"),
    )
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("pct_of_book"))
    .sort("count", descending=True)
)
print("First-time homebuyer breakdown:")
print(fthb)

First-time homebuyer breakdown:
shape: (2, 4)
┌────────────┬─────────┬──────────────────┬─────────────┐
│ FIRST_FLAG ┆ count   ┆ default_rate_pct ┆ pct_of_book │
│ ---        ┆ ---     ┆ ---              ┆ ---         │
│ str        ┆ u32     ┆ f64              ┆ f64         │
╞════════════╪═════════╪══════════════════╪═════════════╡
│ N          ┆ 1558365 ┆ 2.97             ┆ 76.13       │
│ Y          ┆ 488486  ┆ 4.83             ┆ 23.87       │
└────────────┴─────────┴──────────────────┴─────────────┘


## Check for Income or Affordability Fields

Scans the dataset for any column that could carry income, area median income,
or affordability signal. Fannie Mae's public loan performance file is known to
omit borrower income, so this confirms what we do and do not have before
deciding whether an income-based constraint is even possible.

In [19]:
# Search column names for income or affordability-related terms
terms = ["income", "ami", "median", "afford", "lmi", "low_inc", "hud"]

matches = [c for c in df.columns if any(t in c.lower() for t in terms)]
print("Columns matching income/affordability terms:")
print(matches if matches else "None found")

# Also worth a look: the HomeReady program flag, which is Fannie's
# low-to-moderate-income product. Present in your schema as HOMEREADY_PROGRAM_INDICATOR.
print("\nHomeReady program indicator value counts:")
print(df["HOMEREADY_PROGRAM_INDICATOR"].value_counts().sort("count", descending=True))

Columns matching income/affordability terms:
None found

HomeReady program indicator value counts:
shape: (3, 2)
┌─────────────────────────────┬─────────┐
│ HOMEREADY_PROGRAM_INDICATOR ┆ count   │
│ ---                         ┆ ---     │
│ str                         ┆ u32     │
╞═════════════════════════════╪═════════╡
│ 7                           ┆ 1890786 │
│ H                           ┆ 103860  │
│ F                           ┆ 52205   │
└─────────────────────────────┴─────────┘


## Default Rates for Affordable-Program Loans

Compares default rates across the HomeReady program indicator (H = HomeReady,
F = HFA Preferred, 7 = Not applicable). This shows the risk tradeoff of an
affordable-program floor, the same way we checked first-time buyers.

In [20]:
homeready_risk = (
    df.group_by("HOMEREADY_PROGRAM_INDICATOR")
    .agg(
        pl.len().alias("count"),
        (pl.col("default_flag").mean() * 100).round(2).alias("default_rate_pct"),
    )
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("pct_of_book"))
    .sort("count", descending=True)
)
print("Default rate by affordable-program flag:")
print(homeready_risk)

Default rate by affordable-program flag:
shape: (3, 4)
┌─────────────────────────────┬─────────┬──────────────────┬─────────────┐
│ HOMEREADY_PROGRAM_INDICATOR ┆ count   ┆ default_rate_pct ┆ pct_of_book │
│ ---                         ┆ ---     ┆ ---              ┆ ---         │
│ str                         ┆ u32     ┆ f64              ┆ f64         │
╞═════════════════════════════╪═════════╪══════════════════╪═════════════╡
│ 7                           ┆ 1890786 ┆ 3.12             ┆ 92.38       │
│ H                           ┆ 103860  ┆ 5.92             ┆ 5.07        │
│ F                           ┆ 52205   ┆ 9.21             ┆ 2.55        │
└─────────────────────────────┴─────────┴──────────────────┴─────────────┘


## Average Loan Size by Affordable-Program Flag

Compares loan amounts across the HomeReady indicator (H, F, and 7) to see
whether affordable-program loans skew smaller. This informs whether a floor
constraint should be framed as a percent of loan count or a percent of budget
dollars.

In [21]:
size_by_program = (
    df.with_columns(pl.col("ORIG_UPB").cast(pl.Float64, strict=False))
    .group_by("HOMEREADY_PROGRAM_INDICATOR")
    .agg(
        pl.len().alias("count"),
        pl.col("ORIG_UPB").mean().round(0).alias("mean_upb"),
        pl.col("ORIG_UPB").median().round(0).alias("median_upb"),
    )
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("pct_of_book"))
    .sort("count", descending=True)
)
print("Loan size by affordable-program flag:")
print(size_by_program)

Loan size by affordable-program flag:
shape: (3, 5)
┌─────────────────────────────┬─────────┬──────────┬────────────┬─────────────┐
│ HOMEREADY_PROGRAM_INDICATOR ┆ count   ┆ mean_upb ┆ median_upb ┆ pct_of_book │
│ ---                         ┆ ---     ┆ ---      ┆ ---        ┆ ---         │
│ str                         ┆ u32     ┆ f64      ┆ f64        ┆ f64         │
╞═════════════════════════════╪═════════╪══════════╪════════════╪═════════════╡
│ 7                           ┆ 1890786 ┆ 233023.0 ┆ 212000.0   ┆ 92.38       │
│ H                           ┆ 103860  ┆ 181585.0 ┆ 165000.0   ┆ 5.07        │
│ F                           ┆ 52205   ┆ 171340.0 ┆ 161000.0   ┆ 2.55        │
└─────────────────────────────┴─────────┴──────────┴────────────┴─────────────┘


## Findings: Socio-Economic Constraint Options

Explored two dimensions in the data that support a socio-economic floor
constraint: first-time homebuyer status (FIRST_FLAG) and affordable-program
participation (HOMEREADY_PROGRAM_INDICATOR). Borrower income is not present in
the Fannie Mae public file, so these two flags are the usable socio-economic
signals.

### First-Time Homebuyers (FIRST_FLAG)

| Group | Share of Book | Default Rate |
|-------|---------------|--------------|
| First-time buyer (Y) | 23.87% | 4.83% |
| Repeat buyer (N)     | 76.13% | 2.97% |

- Broad, well-supplied signal. Higher risk than repeat buyers, but plenty of
  supply for the optimizer to draw from.

### Affordable Programs (HOMEREADY_PROGRAM_INDICATOR)

| Program | Share of Book | Default Rate | Median Loan |
|---------|---------------|--------------|-------------|
| HomeReady (H)      | 5.07% | 5.92% | $165,000 |
| HFA Preferred (F)  | 2.55% | 9.21% | $161,000 |
| Standard (7)       | 92.38% | 3.12% | $212,000 |

- Smaller, sharper signal. These are income-targeted programs, so they map more
  directly to affordable-lending intent, but carry steeper default risk and
  limited supply.
- Notable finding: HFA Preferred defaults at nearly triple the standard rate,
  meaningfully higher than HomeReady. Not all affordable programs perform the
  same, which is worth reporting rather than hiding by combining them.
- Affordable-program loans run 20 to 25% smaller in balance, so a count-based
  floor consumes less budget than expected. Serving these borrowers costs less
  capacity than it appears.

### Recommendations

- **Keep the two dimensions separate.** First-time buyer and affordable-program
  status capture different things (buyer experience vs income-targeted product),
  and separating them preserves the finding that programs differ in return.
- **Frame floors as a percent of loan count, not budget dollars.** The goal is
  reaching borrowers, which is about people served, not capital deployed. Count
  framing also lets the smaller loan sizes work in your favor.
- **Suggested starting ranges** (to vary in sensitivity analysis, not lock):
  - First-time buyer floor: 25 to 30 percent of funded loans (natural share 23.87 percent)
  - HomeReady (H) floor: 5 to 7 percent of funded loans (natural share 5.07 percent)
  - HFA Preferred (F) floor: 2.5 to 3.5 percent of funded loans (natural share 2.55 percent)

### Suggested Approach

Rather than fixing single values, vary each floor across a small range and
observe how objective of expected return responds. This turns the "not all programs are
equal" insight into a measurable result, showing HFA Preferred costs more return
per unit of floor than HomeReady. That contrast supports the central thesis:
socio-economic consideration and profitability can coexist, with a cost that is
visible and manageable.

## Notebook Summary: Takeaways for Future Stages

### For the ML Stage
- FICO is a strong, clean predictor of default. Rate climbs smoothly from 0.88%
  (Exceptional) to 10.25% (Fair), giving the model real signal.
- FIRST_FLAG carries signal: first-time buyers default at 4.83% vs 2.97% for
  repeat buyers.
- HOMEREADY_PROGRAM_INDICATOR carries signal: HomeReady (H) defaults at 5.92%,
  HFA Preferred (F) at 9.21%, vs 3.12% standard.
- Target is imbalanced at 3.4% default, which the modeling approach must account
  for (metrics, class weighting).

### For the LP Stage
- Budget sizing: mean loan 229k, median 206k, middle 50% between 137k and 300k.
- State concentration matters most for diversification. CA holds 13.7%, but
  concentration and risk do not align (FL, TX, NY are the risky large states;
  CA, WA, CO are safe). This argues for pairing a diversification cap with the
  average-PD ceiling, since each does a different job.
- Zip concentration is diffuse (no prefix above 1.3%), so a zip cap is a soft
  lever compared to state.
- Average-PD ceiling: achievable floor is ~0.88% (safest 10%), rising to 3.5%
  (full book). Meaningful ceiling range is roughly 1% to 3.4%.
- LGD cannot be computed from this data (all recovery fields null across the
  book). Use Sirignano flat values: 30% baseline, 50% downturn stress.
- Interest rates are tightly clustered (mean 4.14%, most between 3.875% and
  4.5%), and rise only ~0.5 points from safest to riskiest FICO band while
  default rises elevenfold. Safe loans likely carry higher expected return, so
  constraints are what force a realistic, diversified portfolio.

### Socio-Economic Constraints (count-based floors, kept separate)
- First-time buyer: 23.87% of book, suggested floor 25-30%.
- HomeReady (H): 5.07% of book, suggested floor 5-7%.
- HFA Preferred (F): 2.55%

## Before and After: One Loan, Many Rows to One Row

To show what the reduction actually does, we take a single loan and follow it
through the process. In the raw file this loan appears once per month, so its
origination facts (like the borrower FICO score) simply repeat, while its
delinquency status changes over time. After the reduction, those monthly rows
collapse into one. The repeating facts are kept once, and the changing
delinquency column becomes two things: the worst delinquency the loan ever
reached, and the final default flag.

In [22]:
import polars as pl

raw_path = "../data/raw/2017Q1.csv"
agg_path = "../data/processed/fannie_2017_loan_level.parquet"

# pull only the five columns we want from the raw monthly file, by position
raw = (
    pl.scan_csv(raw_path, separator="|", has_header=False, infer_schema_length=0)
    .select(
        pl.col("column_2").alias("LOAN_ID"),
        pl.col("column_3").alias("ACT_PERIOD"),
        pl.col("column_24").alias("CSCORE_B"),
        pl.col("column_40").alias("DLQ_STATUS"),
        pl.col("column_44").alias("Zero_Bal_Code"),
    )
)

# find one loan with a long history to use as the example
example_id = (
    raw.group_by("LOAN_ID")
    .agg(pl.len().alias("n_months"))
    .filter(pl.col("n_months") >= 40)
    .head(1)
    .collect(engine="streaming")
)["LOAN_ID"][0]

print(f"Example loan: {example_id}")

Example loan: 133094666195


In [23]:
# BEFORE: the same loan, one row per month (showing first 6 of many)
before = (
    raw.filter(pl.col("LOAN_ID") == example_id)
    .sort("ACT_PERIOD")
    .collect(engine="streaming")
)
print(f"Raw rows for this loan: {before.height}")
before.head(6)

Raw rows for this loan: 107


LOAN_ID,ACT_PERIOD,CSCORE_B,DLQ_STATUS,Zero_Bal_Code
str,str,str,str,str
"""133094666195""","""012018""","""738""","""00""",null
"""133094666195""","""012019""","""738""","""00""",null
"""133094666195""","""012020""","""738""","""00""",null
"""133094666195""","""012021""","""738""","""00""",null
"""133094666195""","""012022""","""738""","""00""",null
"""133094666195""","""012023""","""738""","""00""",null


In [24]:
# AFTER: the same loan, reduced to one row
after = pl.read_parquet(agg_path).filter(pl.col("LOAN_ID") == example_id)
after.select(["LOAN_ID", "CSCORE_B", "max_dlq_ever", "zero_bal_code", "default_flag"])

LOAN_ID,CSCORE_B,max_dlq_ever,zero_bal_code,default_flag
str,str,i32,str,i8
"""133094666195""","""738""",0,null,0
